Scraper de departamentos en **venta** y **alquiler** en Capital Federal.

- Usa `requests` + `BeautifulSoup` (no requiere Playwright)
- Descarga fichas de detalle en **paralelo** (ThreadPoolExecutor)
- Produce `df_venta` y `df_alquiler` con schema compatible con Zonaprop


## 1. Instalación de dependencias

## 2. Imports de utilidades compartidas


In [ ]:
import sys
import os
import re, os, time, json
import requests
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed

sys.path.append(os.path.abspath(".."))

from utils import HEADERS, SCHEMA_COLS, FEATURE_COLS, clean_text, parse_address, parse_price, extract_smart_features, build_output_df, save_df

## 3. URLs y funciones de Argenprop

In [ ]:
BASE_VENTA    = 'https://www.argenprop.com/departamentos/venta/capital-federal'
BASE_ALQUILER = 'https://www.argenprop.com/departamentos/alquiler/capital-federal'

# Paginacion: ?pagina-2, ?pagina-3, ...
# Cada página contiene 20 propiedades

In [ ]:
def _argenprop_detail(url):
    """
    Descarga la página de detalle de un aviso de Argenprop.
    Extrae descripción larga y características estructuradas (si existen).
    """
    out = {
        "Descripción":    "Sin descripción",
        "Caracteristicas":"N/A",
        "Publicado":      "N/A",
        "Visualizaciones": None,
    }
    try:
        r    = requests.get(url, headers=HEADERS, timeout=12)
        soup = BeautifulSoup(r.content, 'html.parser')

        # Descripción larga
        desc_sec = soup.find('section', class_='section-description')
        if desc_sec:
            txt = clean_text(desc_sec.get_text(' ', strip=True))
            txt = re.sub(r'Leer m[aá]s Leer menos', '', txt, flags=re.IGNORECASE).strip()
            if len(txt) > 20:
                out["Descripción"] = txt

        # Características estructuradas (badges/tags en la ficha)
        feat_ul = (
            soup.find('ul', class_='property-features') or
            soup.find('ul', class_='section-icon-features')
        )
        if feat_ul:
            tags = [clean_text(li.get_text(' ', strip=True)) for li in feat_ul.find_all('li')]
            out["Caracteristicas"] = " | ".join(t for t in tags if t and t != "N/A")

    except Exception as e:
        print(f'    [detail error] {e}')
    return out


def scrape_argenprop(base_url: str, mode: str, max_pages: int = 5):
    """
    Scraper principal de Argenprop.

    base_url  : URL base sin paginación.
    mode      : 'venta' o 'alquiler' (solo informativo para el log).
    max_pages : cantidad máxima de páginas a recorrer.

    Devuelve
    -------
    DataFrame con SCHEMA_COLS + FEATURE_COLS, o None si no hay datos.
    """
    all_records = []
    seen_links  = set()

    for page in range(1, max_pages + 1):
        url = f'{base_url}?pagina-{page}' if page > 1 else base_url
        print(f'\n--- Página {page}/{max_pages} [{mode.upper()}] → {url} ---')

        try:
            r    = requests.get(url, headers=HEADERS, timeout=15)
            soup = BeautifulSoup(r.content, 'html.parser')
            items = soup.find_all('div', class_='listing__item')

            if not items:
                print('⚠️  Sin resultados en esta página. Fin.')
                break

            # ── Recolectar datos de tarjetas ──────────────────────────────
            page_cards = []
            for item in items:
                try:
                    link_tag = item.find('a', class_='card')
                    if not link_tag:
                        continue
                    link = 'https://www.argenprop.com' + link_tag['href']
                    if link in seen_links:
                        continue
                    seen_links.add(link)

                    price_el = item.find('p', class_='card__price')
                    precio, expensas = parse_price(
                        clean_text(price_el.text) if price_el else ''
                    )

                    addr_el = item.find('p', class_='card__address')
                    calle, altura, piso = parse_address(
                        clean_text(addr_el.text) if addr_el else 'N/A'
                    )

                    # Barrio / zona (título de la tarjeta, ej. '3 amb. en Palermo')
                    title_el  = (
                        item.find('h2', class_='card__title') or
                        item.find('p',  class_='card__title')
                    )
                    ubicacion = clean_text(title_el.text) if title_el else 'N/A'

                    feat_el  = item.find('ul', class_='card__main-features')
                    detalles = clean_text(feat_el.text) if feat_el else 'N/A'

                    page_cards.append({
                        'link': link, 'Precio': precio, 'Expensas': expensas,
                        'Calle': calle, 'Altura': altura, 'Piso': piso,
                        'Ubicacion': ubicacion, 'Detalles': detalles,
                    })
                except Exception as e:
                    print(f'    [card error] {e}')

            if not page_cards:
                print('Sin tarjetas nuevas. Fin.')
                break

            # ── Descargar fichas en paralelo (mucho más rápido) ───────────
            print(f'  Descargando {len(page_cards)} fichas (paralelo)...')
            detail_map = {}
            with ThreadPoolExecutor(max_workers=8) as pool:
                futs = {pool.submit(_argenprop_detail, c['link']): c['link']
                        for c in page_cards}
                for fut in as_completed(futs):
                    lnk = futs[fut]
                    try:
                        detail_map[lnk] = fut.result()
                    except Exception as e:
                        print(f'    [thread error] {e}')
                        detail_map[lnk] = {}

            for card in page_cards:
                d = detail_map.get(card['link'], {})
                all_records.append({
                    'Fuente':          'Argenprop',
                    'Precio':          card['Precio'],
                    'Expensas':        card['Expensas'],
                    'Calle':           card['Calle'],
                    'Altura':          card['Altura'],
                    'Piso':            card['Piso'],
                    'Ubicacion':       card['Ubicacion'],
                    'Detalles':        card['Detalles'],
                    'Caracteristicas': d.get('Caracteristicas', 'N/A'),
                    'Descripción':     d.get('Descripción',     'Sin descripción'),
                    'Publicado':       d.get('Publicado',       'N/A'),
                    'Visualizaciones': d.get('Visualizaciones', None),
                    'Link':            card['link'],
                })

            print(f'  Nuevas: {len(page_cards)} | Total acumulado: {len(all_records)}')

        except Exception as e:
            print(f'❌ Error en página {page}: {e}')
            break

    return build_output_df(all_records)

print("✅ Funciones de Argenprop listas.")

## 4. Scraping: Departamentos en Venta


In [ ]:
df_venta = scrape_argenprop(BASE_VENTA, 'venta', max_pages=3)

if df_venta is not None:
    print(f'\nTotal venta: {len(df_venta)} propiedades')
    save_df(df_venta, 'argenprop', 'venta')
    display(df_venta.head())

## 5. Scraping: Departamentos en Alquiler

In [ ]:
df_alquiler = scrape_argenprop(BASE_ALQUILER, 'alquiler', max_pages=3)

if df_alquiler is not None:
    print(f'\nTotal alquiler: {len(df_alquiler)} propiedades')
    save_df(df_alquiler, 'argenprop', 'alquiler')
    display(df_alquiler.head())

## 6. Chequeo que se haya scrapeado bien

In [ ]:
for label, df in [('Venta', df_venta), ('Alquiler', df_alquiler)]:
    if df is not None:
        print(f'\n📊 {label} — {len(df)} propiedades')
        pct = (df[FEATURE_COLS].mean() * 100).round(1).astype(str) + '%'
        print(pct.to_string())